# Validation: `plot_roman_availability` and `plot_pa_availability`

This notebook validates the two new functions against the original `RomanPointing` class.

## Expected sun-position offset

`plot_roman_availability` and `plot_pa_availability` compute the Sun's position
with `astropy.coordinates.get_sun`, which gives the **geocentric** direction.
`RomanPointing` uses either an OEM ephemeris for the spacecraft position at L2
or JPL Horizons with JWST as an L2 proxy.  L2 is ~1.5 × 10⁶ km from Earth
(~0.01 AU), so the apparent Sun direction differs by up to **~0.01 AU /
1 AU ≈ 0.01 rad ≈ 0.6°** (worst case, target near quadrature); the actual
measured offset is **~0.24°** for the test date.  All tolerances below are set
with this known systematic in mind.

## What is tested
1. Sun-position offset — measure and document the geocentric vs. L2-proxy gap.
2. Pitch-map spot checks — pitch values match within the sun-offset budget.
3. Observable-zone boundary — visible/invisible flags agree to ≤ 1%.
4. `plot_roman_availability` — smoke test, return types, both frames.
5. `plot_pa_availability` — return types, column names, year coverage.
6. DataFrame pitch column — matches `RomanPointing.get_pitch_angle()`.
7. Nominal PA — matches `RomanPointing.get_position_angle()` within PA budget.
8. PA band width — equals exactly `2 × roll_limit` (internal consistency).
9. PA band edges — match `delta_pitch_roll(0, ±roll_limit)` within PA budget.
10. NaN / invisible-day consistency — structural check, no sun-offset involved.
11. Window transitions — boundary days agree with `RomanPointing`.

In [1]:
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord, get_sun
from astropy import units as u

from roman_opup_tools.roman_attitude import (
    RomanPointing,
    plot_roman_availability,
    plot_pa_availability,
)

# ── shared constants ──────────────────────────────────────────────────────────
TEST_DATE   = '2026-11-21'
TEST_RA     = 269.0    # deg
TEST_DEC    =  66.0    # deg
PITCH_LIMIT = 36.0     # deg
ROLL_LIMIT  = 15.0     # deg
TEST_YEAR   = 2026

# Tolerances — driven by the geocentric vs. L2-proxy sun-position offset.
# The pitch tolerance equals the angular offset between the two Sun positions.
# The PA tolerance is larger because a shift in the sun vector rotates the
# attitude matrix, producing a proportionally larger PA error.
SUN_SEP_TOL   = 0.40   # deg — geocentric vs. L2-proxy sun separation
PITCH_TOL     = 0.40   # deg — propagated into pitch = sun_angle - 90°
PA_TOL        = 1.50   # deg — propagated into position angle
BAND_WIDTH_TOL = 0.50  # deg — roll discretisation (n_rolls=61 → 0.5° step)

def circ_dist(a, b):
    """Minimum angular distance on a circle [0, 360)."""
    d = abs(float(a) - float(b)) % 360.0
    return min(d, 360.0 - d)

PASS = '\033[92m PASS\033[0m'
FAIL = '\033[91m FAIL\033[0m'

def check(condition, name, detail=''):
    tag = PASS if condition else FAIL
    msg = f'{tag}  {name}'
    if detail:
        msg += f'  [{detail}]'
    print(msg)
    return bool(condition)

results = {}
print('Setup complete.')

Setup complete.


---
## 1  Sun-position consistency

Measure the angular separation between the geocentric Sun (`get_sun`) and
`RomanPointing.sun_coord` (L2-proxy or OEM).  This is the root cause of all
subsequent differences and should be below `SUN_SEP_TOL = 0.40°`.

In [2]:
t = Time(TEST_DATE)
pointing = RomanPointing(TEST_DATE)

sun_geo = get_sun(t)
sun_avail = SkyCoord(ra=sun_geo.ra, dec=sun_geo.dec, frame='icrs')
sun_rp    = pointing.sun_coord

sep = sun_avail.separation(sun_rp).deg
print(f'  get_sun (geocentric)   RA={sun_avail.ra.deg:.4f}°  Dec={sun_avail.dec.deg:.4f}°')
print(f'  RomanPointing sun      RA={sun_rp.ra.deg:.4f}°  Dec={sun_rp.dec.deg:.4f}°')
print(f'  Angular separation     {sep:.4f}°  (expected < {SUN_SEP_TOL}°)')
print(f'  Sun source             {pointing._sun_source}')

ok = sep < SUN_SEP_TOL
results['1_sun_position'] = check(ok, f'Sun position agreement < {SUN_SEP_TOL}°',
                                  f'sep={sep:.4f}°')

  get_sun (geocentric)   RA=236.0558°  Dec=-19.7786°
  RomanPointing sun      RA=236.3031°  Dec=-19.7211°
  Angular separation     0.2397°  (expected < 0.4°)
  Sun source             JPL
 PASS  Sun position agreement < 0.4°  [sep=0.2397°]


---
## 2  Pitch-map spot checks

Sample 200 random sky positions, compute pitch two ways, compare.
Expected disagreement ≈ sun-position offset ≈ `PITCH_TOL`.

In [3]:
np.random.seed(42)
ra_s  = np.random.uniform(0, 360, 200)
dec_s = np.random.uniform(-90, 90, 200)

errors = []
for ra_v, dec_v in zip(ra_s, dec_s):
    tgt = SkyCoord(ra=ra_v * u.deg, dec=dec_v * u.deg, frame='icrs')
    pitch_rp  = pointing.get_pitch_angle(target=tgt).value   # uses RomanPointing sun
    pitch_new = sun_avail.separation(tgt).deg - 90.0          # uses get_sun
    errors.append(abs(pitch_new - pitch_rp))

errors = np.array(errors)
print(f'  Max |Δpitch|  = {errors.max():.6f}°  (tol {PITCH_TOL}°)')
print(f'  Mean |Δpitch| = {errors.mean():.6f}°')

ok = errors.max() < PITCH_TOL
results['2_pitch_map'] = check(ok,
    f'Pitch-map spot checks (200 random targets) max error < {PITCH_TOL}°',
    f'max={errors.max():.6f}°')

  Max |Δpitch|  = 0.239690°  (tol 0.4°)
  Mean |Δpitch| = 0.126545°
 PASS  Pitch-map spot checks (200 random targets) max error < 0.4°  [max=0.239690°]


---
## 3  Observable-zone boundary

Compare `visible` flags from both implementations for 300 random positions.
Boundary disagreements are expected only within `PITCH_TOL` of the ±36° pitch
boundary; accept ≤ 1% mismatch.

In [4]:
np.random.seed(7)
ra_s  = np.random.uniform(0, 360, 300)
dec_s = np.random.uniform(-90, 90, 300)

mismatches = 0
for ra_v, dec_v in zip(ra_s, dec_s):
    tgt = SkyCoord(ra=ra_v * u.deg, dec=dec_v * u.deg, frame='icrs')
    pitch_new   = sun_avail.separation(tgt).deg - 90.0
    visible_new = abs(pitch_new) <= PITCH_LIMIT
    pointing.set_target(tgt)
    if visible_new != pointing.is_visible():
        mismatches += 1

rate = mismatches / len(ra_s)
print(f'  Mismatches: {mismatches} / {len(ra_s)}  ({rate*100:.1f}%)')

ok = rate <= 0.01
results['3_observable_zone'] = check(ok, 'Observable-zone agreement ≤ 1% mismatch',
                                     f'{rate*100:.1f}%')

  Mismatches: 0 / 300  (0.0%)
 PASS  Observable-zone agreement ≤ 1% mismatch  [0.0%]


---
## 4  `plot_roman_availability` — smoke tests

In [5]:
import matplotlib.figure

for frame in ('icrs', 'galactic'):
    try:
        fig, ax = plot_roman_availability(
            TEST_DATE, frame=frame,
            pitch_limit=PITCH_LIMIT, roll_limit=ROLL_LIMIT,
        )
        ok_fig = isinstance(fig, matplotlib.figure.Figure)
        ok_ax  = hasattr(ax, 'get_title')
        plt.close('all')
        results[f'4_avail_{frame}_fig']  = check(ok_fig, f'plot_roman_availability ({frame}) returns Figure')
        results[f'4_avail_{frame}_axes'] = check(ok_ax,  f'plot_roman_availability ({frame}) returns Axes')
    except Exception as e:
        plt.close('all')
        results[f'4_avail_{frame}_fig']  = check(False, f'plot_roman_availability ({frame}) smoke test', str(e))
        results[f'4_avail_{frame}_axes'] = False

 PASS  plot_roman_availability (icrs) returns Figure
 PASS  plot_roman_availability (icrs) returns Axes
 PASS  plot_roman_availability (galactic) returns Figure
 PASS  plot_roman_availability (galactic) returns Axes


---
## 5  `plot_pa_availability` — return types & structure

Run once and keep `fig`, `ax`, `df` for all subsequent cells.

In [6]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    fig_pa, ax_pa, df = plot_pa_availability(
        ra=TEST_RA, dec=TEST_DEC,
        year=TEST_YEAR,
        pitch_limit=PITCH_LIMIT,
        roll_limit=ROLL_LIMIT,
        n_days=365,
    )
plt.close('all')

required_cols = {'date', 'pitch_deg', 'visible', 'pa_nominal', 'pa_min', 'pa_max'}
ok_fig  = isinstance(fig_pa, matplotlib.figure.Figure)
ok_ax   = hasattr(ax_pa, 'get_title')
ok_df   = isinstance(df, pd.DataFrame)
ok_cols = required_cols.issubset(df.columns)

print(f'  DataFrame shape  : {df.shape}')
print(f'  Columns present  : {sorted(df.columns.tolist())}')
print(df.head(3).to_string(index=False))

results['5a_pa_fig']  = check(ok_fig,  'plot_pa_availability returns Figure')
results['5b_pa_ax']   = check(ok_ax,   'plot_pa_availability returns Axes')
results['5c_pa_df']   = check(ok_df,   'plot_pa_availability returns DataFrame')
results['5d_pa_cols'] = check(ok_cols, 'DataFrame has all required columns')

  DataFrame shape  : (365, 6)
  Columns present  : ['date', 'pa_max', 'pa_min', 'pa_nominal', 'pitch_deg', 'visible']
      date  pitch_deg  visible  pa_nominal     pa_min     pa_max
2026-01-01  -0.480046     True  168.878480 153.878480 183.878480
2026-01-02  -0.471056     True  167.857199 152.857199 182.857199
2026-01-03  -0.461907     True  166.835933 151.835933 181.835933
 PASS  plot_pa_availability returns Figure
 PASS  plot_pa_availability returns Axes
 PASS  plot_pa_availability returns DataFrame
 PASS  DataFrame has all required columns


### 5e  Year coverage

In [7]:
first_date = pd.Timestamp(df['date'].iloc[0])
last_date  = pd.Timestamp(df['date'].iloc[-1])
ok = (first_date.year == TEST_YEAR and first_date.month == 1 and first_date.day == 1
      and last_date.year == TEST_YEAR)
print(f'  First: {first_date.date()}   Last: {last_date.date()}')
results['5e_year_coverage'] = check(ok, f'DataFrame spans year {TEST_YEAR}')

  First: 2026-01-01   Last: 2026-12-31
 PASS  DataFrame spans year 2026


---
## 6  DataFrame pitch column vs. `RomanPointing.get_pitch_angle()`

Spot-check 30 evenly spaced days.  Expected max error ≈ `PITCH_TOL`.

In [8]:
tgt = SkyCoord(ra=TEST_RA * u.deg, dec=TEST_DEC * u.deg, frame='icrs')
check_idx = np.linspace(0, len(df) - 1, 30, dtype=int)

pitch_errors = []
for idx in check_idx:
    row = df.iloc[idx]
    rp = RomanPointing(row['date'])
    pitch_rp = rp.get_pitch_angle(target=tgt).value
    pitch_errors.append(abs(row['pitch_deg'] - pitch_rp))

pitch_errors = np.array(pitch_errors)
print(f'  Max |Δpitch|  = {pitch_errors.max():.6f}°  (tol {PITCH_TOL}°)')
print(f'  Mean |Δpitch| = {pitch_errors.mean():.6f}°')

ok = pitch_errors.max() < PITCH_TOL
results['6_pa_pitch_series'] = check(ok,
    f'pitch_deg column vs RomanPointing (30 days, tol {PITCH_TOL}°)',
    f'max={pitch_errors.max():.6f}°')

  Max |Δpitch|  = 0.142308°  (tol 0.4°)
  Mean |Δpitch| = 0.082264°
 PASS  pitch_deg column vs RomanPointing (30 days, tol 0.4°)  [max=0.142308°]


---
## 7  Nominal PA vs. `RomanPointing.get_position_angle()`

On observable days, compare `pa_nominal` in the DataFrame against
`get_position_angle()`.  Expected disagreement ≈ `PA_TOL` (sun-vector
difference rotates the attitude matrix).

In [9]:
obs = df[df['visible']]
sample = obs.iloc[np.linspace(0, len(obs) - 1, min(30, len(obs)), dtype=int)]

pa_errors = []
for _, row in sample.iterrows():
    rp = RomanPointing(row['date'])
    rp.set_target(tgt)
    pa_rp = rp.get_position_angle().value % 360
    pa_errors.append(circ_dist(row['pa_nominal'], pa_rp))

pa_errors = np.array(pa_errors)
print(f'  Observable days sampled  : {len(pa_errors)}')
print(f'  Max |ΔPA_nominal|  = {pa_errors.max():.4f}°  (tol {PA_TOL}°)')
print(f'  Mean |ΔPA_nominal| = {pa_errors.mean():.4f}°')

ok = pa_errors.max() < PA_TOL
results['7_pa_nominal'] = check(ok,
    f'Nominal PA vs RomanPointing.get_position_angle() (tol {PA_TOL}°)',
    f'max={pa_errors.max():.4f}°')

  Observable days sampled  : 30
  Max |ΔPA_nominal|  = 1.0837°  (tol 1.5°)
  Mean |ΔPA_nominal| = 0.4473°
 PASS  Nominal PA vs RomanPointing.get_position_angle() (tol 1.5°)  [max=1.0837°]


---
## 8  PA band width = 2 × roll_limit (internal consistency)

This is a purely internal check — both `pa_min` and `pa_max` come from the
same function — so the tolerance here is only the roll-grid discretisation
(`n_rolls=61` → step = 0.5°).

In [10]:
obs = df[df['visible'] & df['pa_min'].notna() & df['pa_max'].notna()]

widths = []
for _, row in obs.iterrows():
    w = row['pa_max'] - row['pa_min']
    if w < 0:
        w += 360.0     # wrapped band
    widths.append(w)

widths = np.array(widths)
expected = 2 * ROLL_LIMIT
errs = np.abs(widths - expected)

print(f'  Observable days with PA data : {len(widths)}')
print(f'  Expected band width          : {expected}°')
print(f'  Mean actual width            : {widths.mean():.4f}°')
print(f'  Max |Δwidth|                 : {errs.max():.4f}°  (tol {BAND_WIDTH_TOL}°)')

ok = errs.max() < BAND_WIDTH_TOL
results['8_pa_band_width'] = check(ok,
    f'PA band width = {expected}° (tol {BAND_WIDTH_TOL}°)',
    f'max_err={errs.max():.4f}°')

  Observable days with PA data : 365
  Expected band width          : 30.0°
  Mean actual width            : 30.0000°
  Max |Δwidth|                 : 0.0000°  (tol 0.5°)
 PASS  PA band width = 30.0° (tol 0.5°)  [max_err=0.0000°]


---
## 9  PA band edges vs. `RomanPointing.delta_pitch_roll(0, ±roll_limit)`

For each sampled observable day, `delta_pitch_roll(0, ±15°)` gives two PA
values; the DataFrame's `{pa_min, pa_max}` should match this pair to within
`PA_TOL`.  Comparison uses circular distance to handle the 0°/360° wrap.

In [11]:
obs = df[df['visible'] & df['pa_min'].notna()]
sample = obs.iloc[np.linspace(0, len(obs) - 1, min(20, len(obs)), dtype=int)]

edge_errors = []
for _, row in sample.iterrows():
    rp = RomanPointing(row['date'])
    rp.set_target(tgt)

    _, _, pa_pos, _ = rp.delta_pitch_roll(0.0,  ROLL_LIMIT)
    _, _, pa_neg, _ = rp.delta_pitch_roll(0.0, -ROLL_LIMIT)
    rp_edges = {pa_pos % 360, pa_neg % 360}

    df_lo = row['pa_min']
    df_hi = row['pa_max']

    # For each DataFrame edge, find the nearest RomanPointing edge
    for df_edge in (df_lo, df_hi):
        best = min(circ_dist(df_edge, rp_e) for rp_e in rp_edges)
        edge_errors.append(best)

edge_errors = np.array(edge_errors)
print(f'  Days sampled     : {len(sample)}')
print(f'  Max edge error   : {edge_errors.max():.4f}°  (tol {PA_TOL}°)')
print(f'  Mean edge error  : {edge_errors.mean():.4f}°')

ok = edge_errors.max() < PA_TOL
results['9_pa_band_edges'] = check(ok,
    f'PA band edges vs delta_pitch_roll (tol {PA_TOL}°)',
    f'max={edge_errors.max():.4f}°')

  Days sampled     : 20
  Max edge error   : 1.0837°  (tol 1.5°)
  Mean edge error  : 0.4500°
 PASS  PA band edges vs delta_pitch_roll (tol 1.5°)  [max=1.0837°]


---
## 10  NaN / invisible-day consistency

Purely structural: `visible=False` rows must have `NaN` PA; `visible=True`
rows must have finite PA values.

In [12]:
pa_cols = ['pa_nominal', 'pa_min', 'pa_max']
invis_all_nan = df[~df['visible']][pa_cols].isna().all(axis=1).all()
vis_all_finite = df[df['visible']][pa_cols].notna().all(axis=1).all()

print(f'  Invisible rows → all NaN  : {invis_all_nan}')
print(f'  Visible rows → all finite : {vis_all_finite}')

results['10a_nan_invisible'] = check(invis_all_nan, 'Invisible days → NaN PA')
results['10b_data_visible']  = check(vis_all_finite, 'Visible days → non-NaN PA')

  Invisible rows → all NaN  : True
  Visible rows → all finite : True
 PASS  Invisible days → NaN PA
 PASS  Visible days → non-NaN PA


---
## 11  Observing-window transition dates vs. `RomanPointing`

For days right at the visible/invisible boundary, `RomanPointing.is_visible()`
should agree with `df['visible']`.  Allow ≤ 1 mismatch (boundary is fuzzy
due to the ~0.24° sun-position offset).

In [13]:
vis_arr = df['visible'].values.astype(int)
transitions = np.where(np.diff(vis_arr) != 0)[0]
check_rows = set()
for t_idx in transitions:
    check_rows.update([max(0, t_idx - 1), t_idx, min(len(df) - 1, t_idx + 1)])

mismatches = 0
for idx in sorted(check_rows):
    row = df.iloc[idx]
    rp = RomanPointing(row['date'])
    rp.set_target(tgt)
    if rp.is_visible() != row['visible']:
        mismatches += 1
        print(f'  boundary mismatch on {row["date"]}  '
              f'df={row["visible"]}  rp={rp.is_visible()}  '
              f'pitch={row["pitch_deg"]:.2f}°')

print(f'  Transition rows checked: {len(check_rows)},  mismatches: {mismatches}')
ok = mismatches <= 1
results['11_window_transitions'] = check(ok,
    'Window transitions agree with RomanPointing (≤ 1 mismatch allowed)',
    f'{mismatches} mismatch(es)')

  Transition rows checked: 0,  mismatches: 0
 PASS  Window transitions agree with RomanPointing (≤ 1 mismatch allowed)  [0 mismatch(es)]


---
## Summary

In [14]:
print('\n' + '═' * 72)
print('VALIDATION SUMMARY')
print('═' * 72)
passed = sum(results.values())
total  = len(results)
for name, ok in results.items():
    tag = '✓' if ok else '✗'
    print(f'  {tag}  {name}')
print('─' * 72)
print(f'  {passed} / {total} passed')
if passed == total:
    print('  All checks passed.')
else:
    print(f'  {total - passed} check(s) FAILED — see cells above for details.')
print('═' * 72)


════════════════════════════════════════════════════════════════════════
VALIDATION SUMMARY
════════════════════════════════════════════════════════════════════════
  ✓  1_sun_position
  ✓  2_pitch_map
  ✓  3_observable_zone
  ✓  4_avail_icrs_fig
  ✓  4_avail_icrs_axes
  ✓  4_avail_galactic_fig
  ✓  4_avail_galactic_axes
  ✓  5a_pa_fig
  ✓  5b_pa_ax
  ✓  5c_pa_df
  ✓  5d_pa_cols
  ✓  5e_year_coverage
  ✓  6_pa_pitch_series
  ✓  7_pa_nominal
  ✓  8_pa_band_width
  ✓  9_pa_band_edges
  ✓  10a_nan_invisible
  ✓  10b_data_visible
  ✓  11_window_transitions
────────────────────────────────────────────────────────────────────────
  19 / 19 passed
  All checks passed.
════════════════════════════════════════════════════════════════════════
